In [1]:
# Change the branch to yours
# And change choose L4 GPU for now in Runtime > Change runtime type
BRANCH = "test-v0.5.6"
REPO_URL = "git@github.com:St1p42/sglang.git"
REPO_DIR = "/content/sglang"

In [2]:
# Generate a public key for SSH - add the printed result to your GitHub account
%%bash
set -e

mkdir -p /root/.ssh
chmod 700 /root/.ssh

if [ ! -f /root/.ssh/id_ed25519 ]; then
  ssh-keygen -t ed25519 -C "colab" -f /root/.ssh/id_ed25519 -N ""
fi

ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
chmod 600 /root/.ssh/known_hosts

echo "Public key:"
cat /root/.ssh/id_ed25519.pub

Public key:
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIGrlb4xb/vBygFgAwSXsIP5QgFgR9SQUfw9bBwNohpjq colab


In [3]:
# Ssh test
%%bash
set -e
ssh -T git@github.com || true

Hi St1p42! You've successfully authenticated, but GitHub does not provide shell access.


In [4]:
import os
os.environ["BRANCH"] = BRANCH
os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = REPO_DIR

In [5]:
# Clone and checkout the branch indicated on the top
# After running, you should be able to see the repo in the Files in the left menu bar
%%bash
set -e

if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
git fetch origin
git checkout "$BRANCH"
git pull origin "$BRANCH"
git branch --show-current
git log --oneline -1

Your branch is up to date with 'origin/test-v0.5.6'.
Already up to date.
test-v0.5.6
7ae368efd chore: bump SGLang version to 0.5.6 (#14316)


Already on 'test-v0.5.6'
From github.com:St1p42/sglang
 * branch                test-v0.5.6 -> FETCH_HEAD


In [6]:
# See GPU/CPU/RAM info
%%bash
echo "=== GPU Info ==="
nvidia-smi || echo "No GPU attached or nvidia-smi failed"

echo -e "\n=== CPU Info ==="
lscpu | grep 'Model name\|Socket(s)\|Core(s) per socket\|Thread(s) per core'

echo -e "\n=== RAM Info ==="
free -h

=== GPU Info ===
Wed Mar 18 04:47:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

In [7]:
# Install
%%bash
set -e
cd /content/sglang

python -m pip uninstall -y sglang sgl-kernel flashinfer-python || true
python -m pip install -e python

Found existing installation: sglang 0.5.6
Uninstalling sglang-0.5.6:
  Successfully uninstalled sglang-0.5.6
Found existing installation: sgl-kernel 0.3.18.post2
Uninstalling sgl-kernel-0.3.18.post2:
  Successfully uninstalled sgl-kernel-0.3.18.post2
Found existing installation: flashinfer-python 0.5.3
Uninstalling flashinfer-python-0.5.3:
  Successfully uninstalled flashinfer-python-0.5.3
Obtaining file:///content/sglang/python
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached flashinfer_python-0.5.3-py3-none-any.whl.metadata (11 kB

In [10]:
# You should see something like:
# torch: 2.9.1+cu128
# sglang version: 0.5.6
# sglang path: ['/content/sglang/python/sglang']
# sglang spec origin: /content/sglang/python/sglang/__init__.py
# sgl_kernel: 0.3.18.post2
# cuda: True NVIDIA L4
import sys

# Make sure Python imports the actual package in your repo
sys.path = [p for p in sys.path if p != "/content/sglang"]
sys.path.insert(0, "/content/sglang/python")

import sglang, torch, sgl_kernel
from importlib.metadata import version

print("torch:", torch.__version__)
print("sglang version:", version("sglang"))
print("sglang path:", list(sglang.__path__))
print("sglang spec origin:", sglang.__spec__.origin)
print("sgl_kernel:", getattr(sgl_kernel, "__version__", "ok"))
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

torch: 2.9.1+cu128
sglang version: 0.5.6
sglang path: ['/content/sglang/python/sglang']
sglang spec origin: /content/sglang/python/sglang/__init__.py
sgl_kernel: 0.3.18.post2
cuda: True NVIDIA L4


In [16]:
# Launch the server (change the model if needed)
%%bash
cd /content/sglang
nohup python -m sglang.launch_server \
  --model-path Qwen/Qwen2.5-1.5B-Instruct \
  --port 30000 \
  > /content/sglang_server.log 2>&1 &

echo $! > /content/sglang_server.pid
sleep 20
tail -n 50 /content/sglang_server.log

2026-03-18 05:26:51.140323: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 05:26:51.166149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773811611.190095   79587 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773811611.199730   79587 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773811611.220174   79587 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [17]:
# Check readiness
%%bash
curl -s http://127.0.0.1:30000/v1/models || true
echo
tail -n 80 /content/sglang_server.log

{"object":"list","data":[{"id":"Qwen/Qwen2.5-1.5B-Instruct","object":"model","created":1773811722,"owned_by":"sglang","root":"Qwen/Qwen2.5-1.5B-Instruct","parent":null,"max_model_len":32768}]}
W0000 00:00:1773811611.220202   79587 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773811611.220204   79587 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773811611.220207   79587 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
2026-03-18 05:26:51.225788: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuil

In [19]:
# Patch so that we don't rely on the vLLM dependency just for the tokenizer
%%bash
set -e
cd /content/sglang/benchmark/multi_turn_chat

python - <<'PY'
from pathlib import Path

p = Path("bench_sglang.py")
s = p.read_text()

s = s.replace(
    "from vllm.transformers_utils.tokenizer import get_tokenizer",
    "from transformers import AutoTokenizer",
)
s = s.replace(
    "tokenizer = get_tokenizer(args.tokenizer, trust_remote_code=args.trust_remote_code)",
    "tokenizer = AutoTokenizer.from_pretrained(args.tokenizer, trust_remote_code=args.trust_remote_code)",
)

p.write_text(s)
print("patched bench_sglang.py")
PY

patched bench_sglang.py


In [20]:
# Run multi-turn chat benchmark
%%bash
set -e
cd /content/sglang/benchmark/multi_turn_chat
python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct

Namespace(turns=4, num_qa=20, min_len_q=256, max_len_q=512, min_len_a=4, max_len_a=8, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=False, long=False, parallel=64, host='http://127.0.0.1', port=30000, backend='srt', device='auto', result_file='result.jsonl', raw_result_file=None)
Latency: 11.536


2026-03-18 05:33:26.443918: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 05:33:26.467737: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773812006.491141   81525 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773812006.498868   81525 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773812006.520735   81525 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [21]:
# Try with 1, 4, and 8 parallel conversations (with a total of 20)
%%bash
cd /content/sglang/benchmark/multi_turn_chat
python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 1
python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 4
python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 8

Namespace(turns=4, num_qa=20, min_len_q=256, max_len_q=512, min_len_a=4, max_len_a=8, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=False, long=False, parallel=1, host='http://127.0.0.1', port=30000, backend='srt', device='auto', result_file='result.jsonl', raw_result_file=None)
Latency: 8.415
Namespace(turns=4, num_qa=20, min_len_q=256, max_len_q=512, min_len_a=4, max_len_a=8, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=False, long=False, parallel=4, host='http://127.0.0.1', port=30000, backend='srt', device='auto', result_file='result.jsonl', raw_result_file=None)
Latency: 3.387
Namespace(turns=4, num_qa=20, min_len_q=256, max_len_q=512, min_len_a=4, max_len_a=8, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=False, long=False, parallel=8, host='http://127.0.0.1', port=30000, backend='srt', device='auto', result_file='result.jsonl', raw_result_file=None)
Latency: 2.277


2026-03-18 05:38:50.320249: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 05:38:50.345216: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773812330.368630   83059 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773812330.376260   83059 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773812330.397856   83059 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Restart the server with custom radix cache + custom cache-aware scheduling (HiCache off)
%%bash
set -e

if [ -f /content/sglang_server.pid ]; then
  kill $(cat /content/sglang_server.pid) || true
  sleep 5
fi

cd /content/sglang
nohup python -m sglang.launch_server \\\n  --model-path Qwen/Qwen2.5-1.5B-Instruct \\\n  --port 30000 \\\n  --radix-cache-impl custom \\\n  --cache-aware-scheduling custom \\\n  > /content/sglang_server_custom.log 2>&1 &

echo $! > /content/sglang_server.pid
sleep 20
curl -s http://127.0.0.1:30000/v1/models || true
echo
tail -n 80 /content/sglang_server_custom.log

cd /content/sglang/benchmark/multi_turn_chat
python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
# Restart the server with custom radix cache + FIFO scheduling baseline (HiCache off)
%%bash
set -e

if [ -f /content/sglang_server.pid ]; then
  kill $(cat /content/sglang_server.pid) || true
  sleep 5
fi

cd /content/sglang
nohup python -m sglang.launch_server \\\n  --model-path Qwen/Qwen2.5-1.5B-Instruct \\\n  --port 30000 \\\n  --radix-cache-impl custom \\\n  --cache-aware-scheduling base \\\n  > /content/sglang_server_fifo.log 2>&1 &

echo $! > /content/sglang_server.pid
sleep 20
curl -s http://127.0.0.1:30000/v1/models || true
echo
tail -n 80 /content/sglang_server_fifo.log

cd /content/sglang/benchmark/multi_turn_chat
python bench_sglang.py --tokenizer Qwen/Qwen2.5-1.5B-Instruct
